In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。也就是Langchain已帮你进行分类了。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据。其实这种类型消息的用处其实就是context。
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        # 使用前提：在头文件的时候导入这些设置
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
        #原来是{"role": "user", "content": "北京今天天气如何？"}
    ]
})

print(response)

In [ ]:
for message in response['messages']:#message是一个消息对象，包含了角色、内容等信息,方便直接遍历
    message.pretty_print()

# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">

主要是基于官方文档中对于多模态输入的处理情况：
https://docs.langchain.com/oss/python/langchain/messages
```python 
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}
```
于是我们后面对于多模态输入的调用是基于此出现的


In [ ]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(#自定义模型
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [ ]:
# 创建Agent
agent = create_agent(model=model)

In [ ]:
# 准备多模态消息，按照前面提到文档的要求
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])
# 相较于原始版本，这里我们直接创建了一个HumanMessage对象，并将文本和图片作为一个列表传入。每个元素都是一个字典，包含了消息的类型和内容。

In [ ]:
stream = agent.stream(
    {"messages": [message]}, #{}是unordered_map；[]是vector，这里是vector<message>，message是一个消息对象，包含了文本和图片等信息
    stream_mode="messages"#返回迭代器的形式
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要**将图片数据转换成base64字符串**，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [ ]:
!uv add ipywidgets

然后，我们创建一个上传组件，用于模拟图片上传。


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件，于是我们直接访问uploader.value[0]，它是一个字典，包含了文件的名称、类型、大小和内容等信息。
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
# 组织多模态消息（来自于挂网的内容）
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)